# 🧠 Обучение модели AI-Весы (EfficientNet-B0 + Transfer Learning на PackEat)

**Версия для лектора** — в ячейках **5–8** есть **TODO** (пропуски `???`), заполняем на уроке.

**КАК ЗАПУСТИТЬ:**
1. Подключите GPU: Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save → **Restart session**.
2. Запускайте ячейки кода последовательно (Shift+Enter).
3. Перед обучением заполните **TODO 1–8**: ячейка 5 (batch), 6 (модель), 7 (loss / optimizer / эпохи), 8 (имя `.pth`).

_Время обучения: ~1.5–2 часа (PackEat + доп. датасет). **21** категория на весах._

### Ячейка 1: Установка и импорты

In [3]:
!pip install kagglehub -q

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets
import json
import os
import shutil
import kagglehub
from tqdm.notebook import tqdm

print(f"PyTorch версия: {torch.__version__}")
print(f"GPU доступен: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем: {device}")


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


PyTorch версия: 2.14.0+cu126
GPU доступен: True
Используем: cuda


### Ячейка 2: Путь к датасету

In [4]:
import os
import torch
from torchvision import datasets, transforms

path = kagglehub.dataset_download("sergeynesteruk/packed-fruits-and-vegetables-recognition-benchmark")

TRAIN_DIR = os.path.join(path, 'train','train')
TEST_DIR = os.path.join(path, 'test','test')

print(f'✅ Пути подтверждены!')


Download already complete (31330405623 bytes).
Extracting files...


OSError: [Errno 28] No space left on device

### Ячейка 3: Объединение сортов PackEat → 21 класс для весов

Сорты из датасета склеиваются в категории, как на экране весов в магазине.  
**Исключены:** яблоко, дыня, слива, мандарин, **абрикос**.

In [1]:
CLASS_MERGE_MAP = {
    'avocado_hass':          'avocado',
    'banana_yellow':         'banana',
    'grape_black':           'grape',
    'grape_green':           'grape',
    'kiwi_kiwi':             'kiwi',
    'lemon_yellow':          'lemon',
    'nectarine_round':       'nectarine',
    'orange_orange':         'orange',
    'pear_conference':       'pear',
    'pomegranate_pomegranate': 'pomegranate',
    'watermelon_elonged':    'watermelon',
    'beet_round':            'beet',
    'cabbage_white':         'cabbage',
    'carrot_carrot':         'carrot',
    'corn_corn':             'corn',
    'cucumber_long':         'cucumber',
    'cucumber_medium':       'cucumber',
    'garlic_garlic':         'garlic',
    'onion_purple':          'onion',
    'onion_white':           'onion',
    'pepper_sweet_green':    'pepper',
    'pepper_sweet_red':      'pepper',
    'pepper_sweet_yellow':   'pepper',
    'potato_yellow':         'potato',
    'tomato_cherry_red':     'tomato',
    'tomato_pink':           'tomato',
    'tomato_slivka':         'tomato',
    'zucchini_green':        'zucchini',
}
selected_classes = sorted(set(CLASS_MERGE_MAP.values()))
NUM_CLASSES = len(selected_classes)
print(f'📦 Исходных папок: {len(CLASS_MERGE_MAP)}')
print(f'📦 После объединения сортов: {NUM_CLASSES} классов')
print(f'   {selected_classes}')

📦 Исходных папок: 28
📦 После объединения сортов: 21 классов
   ['avocado', 'banana', 'beet', 'cabbage', 'carrot', 'corn', 'cucumber', 'garlic', 'grape', 'kiwi', 'lemon', 'nectarine', 'onion', 'orange', 'pear', 'pepper', 'pomegranate', 'potato', 'tomato', 'watermelon', 'zucchini']


### Ячейка 4: Проверка папок

In [5]:
if os.path.exists(TRAIN_DIR):
    folders = sorted(os.listdir(TRAIN_DIR))
    print(f'✅ Папка найдена! Всего классов обнаружено: {len(folders)}')
    print(f'🔎 Примеры классов: {folders[:5]}')
    
    # Проверяем, что все нужные исходные папки на месте
    source_classes = list(CLASS_MERGE_MAP.keys())
    missing = [c for c in source_classes if c not in folders]
    if missing:
        print(f'⚠️ ВНИМАНИЕ: Не найдены папки: {missing}')
    else:
        print(f'✨ Все {len(source_classes)} исходных классов на месте!')
        print(f'📦 После объединения сортов: {NUM_CLASSES} классов')
else:
    print(f'❌ ОШИБКА: Путь {TRAIN_DIR} не найден!')

NameError: name 'TRAIN_DIR' is not defined

### Ячейка 5: Подготовка данных (transforms + DataLoader)

**Data Augmentation** — случайные трансформации (отзеркаливание, поворот, яркость), чтобы модель учила ПРИЗНАКИ, а не запоминала конкретные фото.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

class LabelMapper(Dataset):
    """Перемапливает исходные классы (со сортами) в агрегированные."""
    def __init__(self, dataset, merge_map, transform=None):
        self.dataset = dataset
        self.transform = transform
        self.merge_map = merge_map
        # Новые уникальные классы (без сортов)
        self.new_classes = sorted(set(merge_map.values()))
        self.class_to_idx = {c: i for i, c in enumerate(self.new_classes)}
        # Фильтруем и мапим
        self.samples = []
        for path, old_idx in dataset.samples:
            old_name = dataset.classes[old_idx]
            if old_name in merge_map:
                new_name = merge_map[old_name]
                new_idx = self.class_to_idx[new_name]
                self.samples.append((path, new_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = datasets.folder.default_loader(path)
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print('⌛ Загружаю основной датасет (PackEat)...')
raw_train = datasets.ImageFolder(TRAIN_DIR)
raw_test = datasets.ImageFolder(TEST_DIR)

train_dataset = LabelMapper(raw_train, CLASS_MERGE_MAP, train_transform)
test_dataset = LabelMapper(raw_test, CLASS_MERGE_MAP, test_transform)

# ========== ДОБАВЛЯЕМ ВТОРОЙ ДАТАСЕТ ==========
print('⌛ Скачиваю дополнительный датасет (Fruit & Vegetable)...')
path2 = kagglehub.dataset_download("kritikseth/fruit-and-vegetable-image-recognition")

extra_train_dir = os.path.join(path2, 'train')
extra_test_dir  = os.path.join(path2, 'test')
extra_val_dir   = os.path.join(path2, 'validation')

print(f"   Папки train: {sorted(os.listdir(extra_train_dir))[:5]}...")

# Маппинг второго датасета на НАШИ агрегированные классы
EXTRA_FOLDER_MAP = {
    'banana':        ['banana'],
    'beetroot':      ['beet'],
    'bell pepper':   ['pepper'],
    'cabbage':       ['cabbage'],
    'carrot':        ['carrot'],
    'corn':          ['corn'],
    'sweetcorn':     ['corn'],
    'cucumber':      ['cucumber'],
    'garlic':        ['garlic'],
    'grapes':        ['grape'],
    'kiwi':          ['kiwi'],
    'lemon':         ['lemon'],
    'onion':         ['onion'],
    'orange':        ['orange'],
    'paprika':       ['pepper'],
    'capsicum':      ['pepper'],
    'pear':          ['pear'],
    'pomegranate':   ['pomegranate'],
    'potato':        ['potato'],
    'tomato':        ['tomato'],
    'watermelon':    ['watermelon'],
    'zucchini':      ['zucchini'],
}

def add_extra_samples(target_dataset, extra_dir, folder_map, repeat=5):
    """Добавляет картинки из extra_dir в target_dataset.samples.
    Каждую картинку дублируем repeat раз — transform даст разную аугментацию."""
    added = 0
    for folder, our_classes in folder_map.items():
        folder_path = os.path.join(extra_dir, folder)
        if not os.path.exists(folder_path):
            continue
        images = [f for f in os.listdir(folder_path)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))]
        for i, fname in enumerate(images):
            our_class = our_classes[i % len(our_classes)]
            label = target_dataset.class_to_idx[our_class]
            full_path = os.path.join(folder_path, fname)
            for _ in range(repeat):
                target_dataset.samples.append((full_path, label))
                added += 1
    return added

# Добавляем train + validation второго датасета в наш train (с repeat=5)
added_train  = add_extra_samples(train_dataset, extra_train_dir, EXTRA_FOLDER_MAP, repeat=5)
added_train += add_extra_samples(train_dataset, extra_val_dir,   EXTRA_FOLDER_MAP, repeat=5)
# Добавляем test второго датасета в наш test (test НЕ дублируем)
added_test   = add_extra_samples(test_dataset,  extra_test_dir,  EXTRA_FOLDER_MAP, repeat=1)

print(f'✅ Добавлено из 2-го датасета: {added_train} (train/val), {added_test} (test)')

# Создаём DataLoader
# ═══ 🔶 TODO 1: batch_size ═══
# TODO 1: размер батча — сколько фото обрабатываем за один шаг (обычно 32)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# Генерируем РУССКИЙ class_names.json (агрегированные классы)
translation = {
    "avocado": "Авокадо",
    "banana": "Банан",
    "grape": "Виноград",
    "kiwi": "Киви",
    "lemon": "Лимон",
    "nectarine": "Нектарин",
    "orange": "Апельсин",
    "pear": "Груша",
    "pomegranate": "Гранат",
    "watermelon": "Арбуз",
    "beet": "Свёкла",
    "cabbage": "Капуста",
    "carrot": "Морковь",
    "corn": "Кукуруза",
    "cucumber": "Огурец",
    "garlic": "Чеснок",
    "onion": "Лук",
    "pepper": "Перец болгарский",
    "potato": "Картофель",
    "tomato": "Томат",
    "zucchini": "Кабачок",
}
class_names = {str(i): translation.get(name, name) for i, name in enumerate(selected_classes)}

with open('class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, ensure_ascii=False, indent=4)

print(f'✅ Итого классов: {len(train_dataset.class_to_idx)}')
print(f'📊 Итого фото для обучения: {len(train_dataset)}')
print(f'📊 Итого фото для теста:    {len(test_dataset)}')

### Ячейка 6: Создание модели (Transfer Learning) ⚡ TODO

Здесь 3 шага:
1. **Загрузить** EfficientNet-B0 с весами Google (14 млн картинок!)
2. **Разморозить** все слои — будем дообучать всю модель под наш датасет (fine-tuning)
3. **Заменить** последний слой (1000 → `NUM_CLASSES` классов)

> 💡 EfficientNet-B0 — лёгкая и точная модель (~5.3M параметров, ~15 МБ). Для сравнения, ResNet-50 весит ~100 МБ. Для мобильных устройств можно попробовать `mobilenet_v3_small`, но B0 уже оптимален.

In [ ]:
print("🧠 Создаю модель...")

# ═══ 🔶 TODO 2: модель EfficientNet ═══
# TODO 2: EfficientNet-B0 с весами Google (ImageNet)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# ═══ 🔶 TODO 3: requires_grad ═══
# TODO 3: fine-tuning — обучаем все слои (True) или только голову (False)
for param in model.parameters():
    param.requires_grad = True

# ═══ 🔶 TODO 4: Linear → NUM_CLASSES ═══
# TODO 4: последний слой под NUM_CLASSES (21 класс)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

model = model.to(device)
print(f"✅ Модель готова под {NUM_CLASSES} классов на {device}.")


### Ячейка 7: Обучение ⚡ TODO

Заполните пропуски, затем запускайте целиком (графики loss/accuracy обновятся после каждой эпохи):
- **Loss** — насколько модель ошиблась
- **Optimizer** — как подкручиваются веса
- **NUM_EPOCHS** — сколько проходов по train

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ═══ 🔶 TODO 5: criterion ═══
# TODO 5: функция потерь для классификации (1 из N классов)
criterion = nn.CrossEntropyLoss()

# ═══ 🔶 TODO 6: optimizer ═══
# TODO 6: оптимизатор Adam — все параметры модели, lr=0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# ═══ 🔶 TODO 7: NUM_EPOCHS ═══
# TODO 7: число эпох (для урока: 3)
NUM_EPOCHS = 3

# Хранилище метрик
history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [],
    'test_acc': [],
    'test_loss': []
}

print(f"\n🚀 Начинаю обучение ({NUM_EPOCHS} эпох)...\n")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    batch_losses = []
    batch_indices = []

    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"🚀 Эпоха {epoch+1}/{NUM_EPOCHS}")):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_item = loss.item()
        running_loss += loss_item
        batch_losses.append(loss_item)
        batch_indices.append(batch_idx + 1)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Обновляем график каждые 50 батчей и в конце эпохи
        if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == len(train_loader):
            clear_output(wait=True)
            plt.figure(figsize=(12, 4))

            # График 1: loss по батчам
            plt.subplot(1, 2, 1)
            plt.plot(batch_indices, batch_losses, 'b-', alpha=0.7, label='Train batch loss')
            plt.xlabel('Батч')
            plt.ylabel('Loss')
            plt.title(f'Эпоха {epoch+1}/{NUM_EPOCHS} — Loss по батчам')
            plt.legend()
            plt.grid(True)

            # График 2: accuracy по эпохам (если есть данные)
            plt.subplot(1, 2, 2)
            if history['epoch']:
                plt.plot(history['epoch'], history['train_acc'], 'bo-', label='Train acc')
                plt.plot(history['epoch'], history['test_acc'], 'rx--', markersize=10, label='Test acc')
                plt.xlabel('Эпоха')
                plt.ylabel('Accuracy (%)')
                plt.title('Accuracy по эпохам')
                plt.legend()
                plt.grid(True)
            else:
                plt.text(0.5, 0.5, 'График появится после первой эпохи', ha='center', va='center')
                plt.axis('off')

            plt.tight_layout()
            plt.show()
            print(f"  Эпоха {epoch+1}/{NUM_EPOCHS} | Батч {batch_idx+1}/{len(train_loader)} | Loss: {loss_item:.3f}")

    train_acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)

    # Оценка на тесте
    model.eval()
    test_correct = 0
    test_total = 0
    test_running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="🧪 Валидация", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_acc = 100 * test_correct / test_total
    test_avg_loss = test_running_loss / len(test_loader)

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)
    history['test_loss'].append(test_avg_loss)

    # Финальный график эпохи с крестиком валидации
    clear_output(wait=True)
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(batch_indices, batch_losses, 'b-', alpha=0.6, label='Train batch loss')
    plt.scatter([len(batch_indices)], [test_avg_loss], color='red', marker='x', s=100, zorder=5, label=f'Test loss: {test_avg_loss:.3f}')
    plt.xlabel('Батч')
    plt.ylabel('Loss')
    plt.title(f'Эпоха {epoch+1} — Loss по батчам + Test loss (X)')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['epoch'], history['train_acc'], 'bo-', label='Train acc')
    plt.plot(history['epoch'], history['test_acc'], 'rx--', markersize=10, label='Test acc')
    plt.xlabel('Эпоха')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy по эпохам')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    print(f"  Эпоха {epoch+1}/{NUM_EPOCHS} — Train Loss: {avg_loss:.3f} | Train: {train_acc:.1f}% | Test: {test_acc:.1f}%")

# Итоговые графики
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['epoch'], history['train_loss'], 'bo-', label='Train loss')
plt.plot(history['epoch'], history['test_loss'], 'rx--', markersize=10, label='Test loss')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Loss по эпохам')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['epoch'], history['train_acc'], 'bo-', label='Train acc')
plt.plot(history['epoch'], history['test_acc'], 'rx--', markersize=10, label='Test acc')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy по эпохам')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\n🎉 Обучение завершено! Лучшая точность на тесте: {max(history['test_acc']):.1f}%")

### Ячейка 8: Сохранение модели ⚡ TODO

In [ ]:
# ═══ 🔶 TODO 8: MODEL_PATH ═══
# TODO 8: имя файла с весами модели
MODEL_PATH = "product_model.pth"

model = model.to("cpu")
torch.save(model.state_dict(), MODEL_PATH)

file_size = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"✅ Модель сохранена: {MODEL_PATH} ({file_size:.1f} МБ)")
print(f"✅ class_names.json сохранён")
print(f"\n📥 Скачайте файл и положите в папку 1_ml/:")
print(f"   • {MODEL_PATH}")
print(f"\n💡 В Colab: правый клик на файл → Download")

### Ячейка 8.5: Матрица ошибок (Confusion Matrix) и метрики классификации

Посмотрим, на каких классах модель ошибается чаще всего. Для этого построим матрицу ошибок и посчитаем precision / recall / F1-score.

> 📊 **Accuracy** — доля правильных ответов.  
> 🎯 **Precision** — из всех предсказаний класса «X», сколько были правильными.  
> 🔍 **Recall** — из всех реальных фото класса «X», сколько модель нашла.  
> ⚖️ **F1-score** — среднее между precision и recall.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model = model.to(device)
# Собираем все предсказания на тесте
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="📊 Сбор предсказаний", leave=False):
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Список русских названий классов по порядку
class_labels = [class_names[str(i)] for i in range(NUM_CLASSES)]

# --- Confusion Matrix ---
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(18, 16))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.5, linecolor='gray')
plt.title("Матрица ошибок (Confusion Matrix)", fontsize=16)
plt.xlabel("Предсказанный класс", fontsize=12)
plt.ylabel("Истинный класс", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# --- Classification Report ---
print("\n" + "="*60)
print("📋 Отчёт по классификации (Classification Report)")
print("="*60)
report = classification_report(all_labels, all_preds,
                                target_names=class_labels,
                                digits=3)
print(report)

### Ячейка 9: Быстрый тест

In [ ]:
from PIL import Image
import random

model.eval()
test_img_path = random.choice(test_dataset.samples)[0]
img = Image.open(test_img_path).convert("RGB")
input_tensor = test_transform(img).unsqueeze(0)

with torch.no_grad():
    outputs = model(input_tensor.to(device))
    probs = torch.nn.functional.softmax(outputs[0], dim=0)
    top3_prob, top3_idx = torch.topk(probs, 3)

print("🧪 Быстрый тест (случайное фото из теста):")
for i in range(3):
    name = class_names[str(top3_idx[i].item())]
    conf = top3_prob[i].item() * 100
    medal = ["🥇", "🥈", "🥉"][i]
    print(f"  {medal} {name}: {conf:.1f}%")

### Ячейка 10: Тест на своей картинке

In [ ]:
from google.colab import files
from PIL import Image
import torch

print("Выберите фото продукта с вашего компьютера:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n✅ Файл {filename} загружен! Анализирую...")
    img = Image.open(filename).convert("RGB")
    input_tensor = test_transform(img).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        outputs = model(input_tensor.to(device))
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
        top3_prob, top3_idx = torch.topk(probs, 3)

    print("\n🧪 Результат:")
    for i in range(3):
        name = class_names[str(top3_idx[i].item())]
        conf = top3_prob[i].item() * 100
        medal = ["🥇", "🥈", "🥉"][i]
        print(f"  {medal} {name}: {conf:.1f}%")

### Ячейка 11: Тест на 10 случайных картинках с топ-5 предсказаний

Посмотрим, как модель ведёт себя на реальных фото. Для каждой картинки покажем:
- само фото
- горизонтальный bar chart с уверенностью для топ-5 классов

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import random
import torch.nn.functional as F

model.eval()

# Выбираем 10 случайных из теста
samples = random.sample(test_dataset.samples, 10)

fig, axes = plt.subplots(10, 2, figsize=(14, 30))
fig.suptitle('Тест на 10 случайных картинках', fontsize=16, y=0.995)

for row, (img_path, true_label) in enumerate(tqdm(samples, desc="📊 Анализ 10 картинок")):
    # Загружаем картинку
    img = Image.open(img_path).convert("RGB")
    input_tensor = test_transform(img).unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_tensor.to(device))
        probs = F.softmax(outputs[0], dim=0)
        top5_prob, top5_idx = torch.topk(probs, 5)

    # --- Левая колонка: изображение ---
    ax_img = axes[row, 0]
    ax_img.imshow(img)
    true_name = class_names[str(true_label)]
    pred_name = class_names[str(top5_idx[0].item())]
    color_title = 'green' if true_label == top5_idx[0].item() else 'red'
    ax_img.set_title(f'Истина: {true_name}\nПредсказание: {pred_name}', color=color_title, fontsize=11)
    ax_img.axis('off')

    # --- Правая колонка: bar chart топ-5 ---
    ax_bar = axes[row, 1]
    names = [class_names[str(idx.item())] for idx in top5_idx]
    values = [p.item() * 100 for p in top5_prob]
    colors = ['#2ca02c' if idx.item() == true_label else '#1f77b4' for idx in top5_idx]

    y_pos = np.arange(len(names))
    ax_bar.barh(y_pos, values, color=colors, edgecolor='black')
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(names, fontsize=10)
    ax_bar.invert_yaxis()
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xlabel('Уверенность (%)')
    for i, v in enumerate(values):
        ax_bar.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Ячейка 12: Тест на своей картинке (локально / для продакшена)

Если вы запускаете НЕ в Colab, а на своём компьютере или сервере — используйте эту ячейку.
Просто укажите путь к файлу в переменной `MY_IMAGE_PATH`.

In [ ]:
MY_IMAGE_PATH = "my_photo.jpg"  # <-- ЗАМЕНИТЕ на путь к вашему файлу

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Загружаем и предсказываем
img = Image.open(MY_IMAGE_PATH).convert("RGB")
input_tensor = test_transform(img).unsqueeze(0)

model.eval()
with torch.no_grad():
    outputs = model(input_tensor)
    probs = torch.nn.functional.softmax(outputs[0], dim=0)
    top5_prob, top5_idx = torch.topk(probs, 5)

# Визуализация
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4))

ax_img.imshow(img)
ax_img.set_title(f'Ваше фото: {MY_IMAGE_PATH}')
ax_img.axis('off')

names = [class_names[str(idx.item())] for idx in top5_idx]
values = [p.item() * 100 for p in top5_prob]
colors = ['#2ca02c' if i == 0 else '#1f77b4' for i in range(len(names))]

y_pos = np.arange(len(names))
ax_bar.barh(y_pos, values, color=colors, edgecolor='black')
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(names, fontsize=11)
ax_bar.invert_yaxis()
ax_bar.set_xlim(0, 100)
ax_bar.set_xlabel('Уверенность (%)')
for i, v in enumerate(values):
    ax_bar.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=10)
ax_bar.set_title('Топ-5 предсказаний')

plt.tight_layout()
plt.show()

print("\n🧪 Результат:")
for i in range(3):
    name = class_names[str(top5_idx[i].item())]
    conf = top5_prob[i].item() * 100
    medal = ["🥇", "🥈", "🥉"][i]
    print(f"  {medal} {name}: {conf:.1f}%")

### Ячейка 13: Как использовать модель на продакшене (Python / сервер / приложение)

Чтобы запустить обученную модель у себя в проекте, нужно сделать 5 простых шагов:

1. **Скопируйте файлы**  
   Вам понадобятся 2 файла (сохранённые ранее):  
   - `product_model.pth` — веса модели  
   - `class_names.json` — словарь классов  

2. **Создайте модель и загрузите веса**  
   ```python
   import torch
   from torchvision import models

   NUM_CLASSES = len(class_names)  # 21 после объединения сортов
   model = models.efficientnet_b0(weights=None)
   model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
   model.load_state_dict(torch.load("product_model.pth", map_location="cpu"))
   model.eval()  # обязательно — отключает Dropout и BatchNorm
   ```

3. **Подготовьте трансформации**  
   Картинка ДОЛЖНА проходить ТЕ ЖЕ преобразования, что и при обучении:  
   ```python
   from torchvision import transforms

   transform = transforms.Compose([
       transforms.Resize((224, 224)),
       transforms.ToTensor(),
       transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
   ])
   ```

4. **Сделайте предсказание**  
   ```python
   from PIL import Image

   img = Image.open("фото.jpg").convert("RGB")
   tensor = transform(img).unsqueeze(0)  # добавляем batch dimension

   with torch.no_grad():
       outputs = model(tensor)
       probs = torch.nn.functional.softmax(outputs[0], dim=0)
   ```

5. **Превратите индекс в название**  
   Загрузите `class_names.json` и берите название по индексу с наибольшей вероятностью.  

> ⚠️ **Важно:**  
> - Размер входа — строго **224×224 пикселя**.  
> - Значения пикселей нужно нормализовать теми же `mean` и `std`.  
> - Модель ожидает тензор формы `(1, 3, 224, 224)` — поэтому `unsqueeze(0)`.  
> - На проде используйте `torch.no_grad()` — это отключает вычисление градиентов и ускоряет инференс.